In [ ]:
pip install pdfplumber pandas openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pdfplumber
import pandas as pd
import re

pdf_file = "488 page dataset.pdf"
all_rows = []

with pdfplumber.open(pdf_file) as pdf:
    for page_num, page in enumerate(pdf.pages):
        table = page.extract_table(
            {
                "vertical_strategy": "lines",
                "horizontal_strategy": "lines",
                "snap_tolerance": 5,
                "join_tolerance": 5,
                "edge_min_length": 3,
                "min_words_vertical": 1,
                "min_words_horizontal": 1,
            }
        )
        if not table:
            continue
        for row in table:
            if not row:
                continue
            cleaned_row = []
            for cell in row:
                if cell is None:
                    cleaned_row.append("")
                    continue            
                cell = re.sub(r"\s*\n\s*", " ", str(cell))
                cell = re.sub(r"\s+", " ", cell).strip()
                cleaned_row.append(cell)
            all_rows.append(cleaned_row)
df = pd.DataFrame(all_rows)
df.to_csv("flora_extracted_improved.csv", index=False)
print("Done")

Done


In [ ]:
import pandas as pd

# 1. Load your raw CSV file
csv_file = "flora_extracted_improved.csv"
df = pd.read_csv(csv_file, header=None)

# 2. Define your clean headers without any commas
clean_headers = [
    "Number",
    "Scientific Name",
    "Common Names",
    "Plant Division",
    "Plant Growth Form",
    "Lifespan (in Singapore)",
    "Mode of Nutrition",
    "Plant Shape",
    "Maximum Height From",
    "Maximum Height To",
]

# 3. Clean up the rows
# Remove rows that contain "Scientific Name" or "Scientific Name," in column index 1
df_cleaned = df[~df[1].astype(str).str.contains("Scientific Name")].copy()

# Remove the row that just has numbers (like row 1 in your picture: 0, 1, 2, 3...)
df_cleaned = df_cleaned[df_cleaned[0] != "0"]
df_cleaned = df_cleaned[df_cleaned[0] != 0]

# 4. Clear any completely blank/empty rows
df_cleaned.dropna(how="all", inplace=True)

# 5. Set your new clean headers at the top
df_cleaned.columns = clean_headers

# 6. Save back to a single, clean CSV file
df_cleaned.to_csv("flora_final_cleaned.csv", index=False)

print(f"All repeating headers removed! Cleaned data saved. Rows remaining: {len(df_cleaned)}")

All repeating headers removed! Cleaned data saved. Rows remaining: 3170


In [ ]:
import re
import pandas as pd

# Load data
df = pd.read_csv("flora_final_cleaned.csv")


def clean_citations(name):
    if pd.isna(name):
        return name
    name = str(name).strip()

    # 1. Remove parenthetical citations and everything after them (e.g., "(L.) Moench")
    name = re.sub(r"\s*\([^)]+\)\s*.*", "", name)

    # 2. Remove lone authority names/abbreviations at the end (e.g., "Willd.", "F.Muell.", "L.")
    name = re.sub(
        r"\s+[A-Z][a-z]*\.?(?:\s*&\s*[A-Z][a-z]*\.?)?(?:\s+ex\s+[A-Z][a-z]*\.?)?$",
        "",
        name,
    )

    # 3. Clean up loose ends like 'cultivars'
    name = re.sub(r"\s+cultivars?$", "", name, flags=re.IGNORECASE)

    return name.strip()


# Apply specific citation cleaning
df["Scientific Name"] = df["Scientific Name"].apply(clean_citations)

# Save progress
df.to_csv("flora_final_cleaned1.csv", index=False)
print("Code 1 Complete: Citations removed from Scientific Names.")

Code 1 Complete: Citations removed from Scientific Names.


In [ ]:
def clean_commas(text):
    if pd.isna(text): return text
    text = str(text).strip()
    # 1. Compress multi-comma anomalies into unified single marks
    text = re.sub(r",\s*,+", ",", text)
    # 2. Enforce standard trailing post-comma single space constraints
    text = re.sub(r"\s*,\s*", ", ", text)
    # 3. Right-strip dangling margins
    text = text.rstrip(", ")
    if text in [",", "", "nan"]: return None
    return text

df = pd.read_csv("flora_final_cleaned1.csv")
for col in df.columns:
    df[col] = df[col].apply(clean_commas)
df.to_csv("flora_final_cleaned2.csv", index=False)

Code 2 Complete: Double and trailing commas completely cleaned.


In [5]:
import re
import pandas as pd

# 1. Load your clean data
df = pd.read_csv("flora_final_cleaned2.csv")

# Create a clean working DataFrame for the new topic schema
df_mapped = pd.DataFrame()

# Direct copy text columns
df_mapped["Scientific_Name"] = df["Scientific Name"]
df_mapped["Common_Name"] = df["Common Names"]


# --- HEURISTIC 1: Mature Height (Converted strictly to cm) ---
def parse_height_to_cm(height_val):
    if pd.isna(height_val):
        return 100  # Fallback default if empty
    numbers = re.findall(r"\d+\.?\d*", str(height_val))
    if numbers:
        val = float(numbers[0])
        if "m" in str(height_val).lower() or val < 100:
            return int(val * 100)
        return int(val)
    return 100


df_mapped["Mature_Height_cm"] = df["Maximum Height To"].apply(parse_height_to_cm)


# --- HEURISTIC 2: Strict Categories Setup (9 Distinct Classes) ---
def map_strict_category(form, division):
    form_str = str(form).lower() if pd.notna(form) else ""
    div_str = str(division).lower() if pd.notna(division) else ""

    if "fern" in form_str or "fern" in div_str:
        return "Ferns"
    elif "palm" in form_str:
        return "Palms"
    elif "climber" in form_str or "vine" in form_str:
        return "Climbers & Vines"
    elif "tree" in form_str:
        return "Trees"
    elif "shrub" in form_str:
        return "Shrubs"
    elif "succulent" in form_str or "cactus" in form_str:
        return "Succulents & Cacti"
    elif (
        "herb" in form_str
        or "vegetable" in form_str
        or "edible" in form_str
        or "medicinal" in form_str
    ):
        return "Edibles & Herbs"
    elif "flower" in form_str or "orchid" in form_str:
        return "Flowering Ornamentals"
    else:
        return "Foliage Plants"  # Catch-all default for leafy greens


df_mapped["Categories"] = df.apply(
    lambda r: map_strict_category(r["Plant Growth Form"], r["Plant Division"]),
    axis=1,
)


# --- HEURISTIC 3: Space Required ---
def map_space(height):
    if height < 50:
        return "Small"
    elif height <= 200:
        return "Medium"
    else:
        return "Large"


df_mapped["Space_Required"] = df_mapped["Mature_Height_cm"].apply(map_space)


# --- HEURISTIC 4: House Type (Exact user matching constraints) ---
def map_exact_house(category, space_req):
    # Trees and massive shrubs go to Landed property
    if category == "Trees" or (category == "Shrubs" and space_req == "Large"):
        return "Landed (Terrace/Semi-D/Bungalow)"
    # Sprawling climbers and climbers need sprawling or village fencing setups
    elif category == "Climbers & Vines":
        return "Tradisional/Spacious (Village House)"
    # Compact plants fit vertical/indoor spaces perfectly
    else:
        return "High-rise (Apartment/Flat)"


df_mapped["House_Type"] = df_mapped.apply(
    lambda r: map_exact_house(r["Categories"], r["Space_Required"]), axis=1
)


# --- HEURISTIC 5: Gardening Level (Mapped across all 9 categories) ---
def map_gardening_level(category):
    # Easiest structural plants to maintain
    if category in ["Trees", "Shrubs", "Palms"]:
        return "Beginner"
    # Need regular trimming, soil tracking, or medium attention
    elif category in ["Foliage Plants", "Climbers & Vines", "Succulents & Cacti"]:
        return "Moderate"
    # Needs perfect moisture, seasonal pruning, or food propagation care
    else:  # Flowering Ornamentals, Edibles & Herbs, Ferns
        return "Expert"


df_mapped["Gardening_Level"] = df_mapped["Categories"].apply(map_gardening_level)


# --- HEURISTIC 6: Light Condition (3 Distinct Options: full sun, partial sun, low light) ---
def map_light_condition(category):
    if category in ["Trees", "Succulents & Cacti", "Palms"]:
        return "Full Sun"
    elif category in ["Shrubs", "Climbers & Vines", "Edibles & Herbs"]:
        return "Partial Sun"
    else:  # Ferns, Foliage Plants, Flowering Ornamentals
        return "Low Light"


df_mapped["Light_Condition"] = df_mapped["Categories"].apply(map_light_condition)


# --- HEURISTIC 7: Water Frequency (3 Distinct Options: daily, weekly, monthly) ---
def map_water_frequency(category):
    if category in ["Ferns", "Edibles & Herbs", "Flowering Ornamentals"]:
        return "Daily"
    elif category in [
        "Trees",
        "Shrubs",
        "Palms",
        "Climbers & Vines",
        "Foliage Plants",
    ]:
        return "Weekly"
    else:  # Succulents & Cacti
        return "Monthly"


df_mapped["Water_Frequency"] = df_mapped["Categories"].apply(map_water_frequency)


# --- HEURISTIC 8: Growth Rate (3 Distinct Options: fast, moderate, slow) ---
def map_growth_rate(category):
    if category in ["Climbers & Vines", "Edibles & Herbs"]:
        return "Fast"
    elif category in ["Shrubs", "Flowering Ornamentals", "Foliage Plants", "Ferns"]:
        return "Moderate"
    else:  # Trees, Palms, Succulents & Cacti
        return "Slow"


df_mapped["Growth_Rate"] = df_mapped["Categories"].apply(map_growth_rate)


# --- HEURISTIC 9: Humidity Need (3 Distinct Options: low, medium, high) ---
def map_humidity_need(category):
    if category in ["Ferns", "Foliage Plants"]:
        return "High"
    elif category in ["Succulents & Cacti"]:
        return "Low"
    else:  # Trees, Shrubs, Palms, Climbers & Vines, Edibles & Herbs, Flowering Ornamentals
        return "Medium"


df_mapped["Humidity_Need"] = df_mapped["Categories"].apply(map_humidity_need)


# --- FINAL RE-ORDERING TO MATCH YOUR EXACT REQUEST SCHEMA ---
final_column_order = [
    "Scientific_Name",
    "Common_Name",
    "House_Type",
    "Gardening_Level",
    "Space_Required",
    "Light_Condition",
    "Categories",
    "Water_Frequency",
    "Growth_Rate",
    "Mature_Height_cm",
    "Humidity_Need",
]
df_mapped = df_mapped[final_column_order]

# Save back into clean topic-focused CSV
df_mapped.to_csv("flora_topic_focused.csv", index=False)

print("Strict Heuristic Mapping Applied Successfully!")
# Show preview matrix inside your notebook cell
df_mapped.head(10)

Strict Heuristic Mapping Applied Successfully!


,Scientific_Name,Common_Name,House_Type,Gardening_Level,Space_Required,Light_Condition,Categories,Water_Frequency,Growth_Rate,Mature_Height_cm,Humidity_Need
0,Abelia × grandiflora,Glossy abelia,High-rise (Apartment/Flat),Beginner,Medium,Partial Sun,Shrubs,Weekly,Moderate,100,Medium
1,Abelia × grandiflora 'Francis Mason',"Golden Abelia, Francis Mason Glossy Abelia",High-rise (Apartment/Flat),Beginner,Medium,Partial Sun,Shrubs,Weekly,Moderate,100,Medium
2,Abelia × grandiflora 'Kaleidosco pe',Kaleidoscop e Abelia,Landed (Terrace/Semi-D/Bungalow),Beginner,Large,Partial Sun,Shrubs,Weekly,Moderate,7000,Medium
3,Abelmoschu s esculentus,"Lady's Fingers, Okra, Gumbo, Bendi",High-rise (Apartment/Flat),Expert,Medium,Partial Sun,Edibles & Herbs,Daily,Fast,180,Medium
4,Abelmoschu s esculentus 'Alabama Red',Lady's Finger,High-rise (Apartment/Flat),Expert,Medium,Partial Sun,Edibles & Herbs,Daily,Fast,100,Medium
5,Abelmoschu s esculentus 'Clemson Spineless',"Lady's Finger, Okra",High-rise (Apartment/Flat),Expert,Medium,Partial Sun,Edibles & Herbs,Daily,Fast,100,Medium
6,Abelmoschu s esculentus,Lady's Finger,High-rise (Apartment/Flat),Expert,Medium,Partial Sun,Edibles & Herbs,Daily,Fast,100,Medium
7,Abelmoschu s esculentus 'Greeny Splendor',Lady's Finger,High-rise (Apartment/Flat),Expert,Medium,Partial Sun,Edibles & Herbs,Daily,Fast,100,Medium
8,Abelmoschu s esculentus 'Silver Queen',"Lady's Finger, Okra",High-rise (Apartment/Flat),Expert,Medium,Partial Sun,Edibles & Herbs,Daily,Fast,180,Medium
9,Abelmoschu s esculentus 'Star of David',"Lady's Finger, Okra",High-rise (Apartment/Flat),Expert,Medium,Partial Sun,Edibles & Herbs,Daily,Fast,100,Medium


In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# 1. Load your original unencoded dataset using the file you provided
df = pd.read_csv("PlantMatch_Ultimate_Complete - Copy.csv")

# Initialize clean feature matrix dataframe
encoded_df = pd.DataFrame(index=df.index)

# Place sequential ID as the absolute first column on the left side
encoded_df["ID"] = range(1, len(df) + 1)

# ==========================================================
# STEP A: ORDINAL MAPPING
# ==========================================================
ordinal_mappings = {
    "Gardening_Level": {"Beginner": 1, "Moderate": 2, "Expert": 3},
    "Space_Required": {"Small": 1, "Medium": 2, "Large": 3},
    "Light_Condition": {"Low Light": 1, "Partial Sun": 2, "Partial Shade": 2, "Full Sun": 3},
    "Water_Frequency": {"Monthly": 1, "Weekly": 2, "Daily": 3},
    "Growth_Rate": {"Slow": 1, "Moderate": 2, "Fast": 3},
    "Humidity_Need": {"Low": 1, "Medium": 2, "High": 3},
}

for col, mapping in ordinal_mappings.items():
    encoded_df[col] = df[col].map(mapping)

# ==========================================================
# STEP B: GLOBAL MIN-MAX NORMALIZATION
# ==========================================================
scaler = MinMaxScaler()
# Scale only the ordinal features mapped above
columns_to_scale = [c for c in encoded_df.columns if c != "ID"]
encoded_df[columns_to_scale] = scaler.fit_transform(encoded_df[columns_to_scale])

# Directly attach your pre-normalized mature height data without rescaling it
encoded_df["Mature_Height_cm"] = df["Mature_Height_cm"]

# ==========================================================
# STEP C: ONE-HOT ENCODING
# ==========================================================
# Clean category strings to prevent encoding white spaces
df["Categories"] = df["Categories"].str.strip()

# Standardize the 3 house types explicitly before encoding to prevent text mismatches
house_map = {
    "High-rise (Apartment/Flat)": "High-rise (Apartment/Flat)",
    "Landed (Terrace/Semi-D/Bungalow)": "Landed (Terrace/Semi-D/Bungalow)",
    "Tradisional/Spacious (Village House)": "Tradisional/Spacious (Village House)"
}
df["House_Type_Cleaned"] = df["House_Type"].str.strip().map(house_map)

# Perform one-hot encoding explicitly using your exact string structures
ohe_categories = pd.get_dummies(df["Categories"], prefix="Categories", dtype=float)
ohe_house = pd.get_dummies(df["House_Type_Cleaned"], prefix="House_Type", dtype=float)

# Combine one-hot columns together
ohe_columns = pd.concat([ohe_categories, ohe_house], axis=1)
encoded_df = pd.concat([encoded_df, ohe_columns], axis=1)

# Move descriptive text blocks safely to the front, right after the ID column
encoded_df.insert(1, "Scientific_Name", df["Scientific_Name"])
encoded_df.insert(2, "Common_Name", df["Common_Name"])

# ==========================================================
# STEP D: APPEND IMAGE URLS AT THE END
# ==========================================================
encoded_df["Image_URL"] = df["Image_URL"]

# Save out finalized matrix
encoded_df.to_csv("final_encoded_plant_features.csv", index=False)
print("Encoding complete! Light condition mapping missing values successfully fixed.")

Encoding complete! Light condition mapping missing values successfully fixed.


In [11]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load your perfectly preprocessed final matrix
df_encoded = pd.read_csv("final_encoded_plant_features.csv")

# Isolate numeric columns for the formula (skipping text identifiers)
math_features = df_encoded.iloc[:, 2:]

# Compute Pairwise Cosine Similarity matrix
cosine_sim_matrix = cosine_similarity(math_features)

# Convert math array into a clean searchable DataFrame mapped to Scientific Names
similarity_repo = pd.DataFrame(
    cosine_sim_matrix, 
    index=df_encoded["Scientific_Name"], 
    columns=df_encoded["Scientific_Name"]
)
print("Cosine Similarity Lookup Repository successfully generated!")

# ==========================================================
# UPDATED FUNCTION: KEEPS BOTH DECIMAL AND %
# ==========================================================
def get_content_recommendations_dual_score(target_plant_scientific, top_n=5):
    """
    Input a plant's scientific name, and this function returns the top matches
    showing both the raw mathematical decimal score and the UI percentage match.
    """
    if target_plant_scientific not in similarity_repo.index:
        return f"Error: '{target_plant_scientific}' is not found in the dataset."
        
    # Grab the row of similarity scores for our target plant
    plant_scores = similarity_repo[target_plant_scientific]
    
    # Sort downward (highest score first) and slice top rows (skipping index 0)
    top_matches = plant_scores.sort_values(ascending=False).iloc[1:top_n+1]
    
    # Create the results table holding BOTH values
    recommendations_df = pd.DataFrame({
        "Scientific_Name": top_matches.index,
        "Similarity_Score_Decimal": top_matches.values,  # Keep the raw math decimal
        # Convert decimal to human-readable string percentage formatted to 1 decimal place
        "Match_Percentage": [f"{round(score * 100, 1)}%" for score in top_matches.values] 
    })
    
    # Merge back with original dataset to fetch the human-readable Common Name
    final_output = recommendations_df.merge(
        df_encoded[["Scientific_Name", "Common_Name"]], 
        on="Scientific_Name", 
        how="left"
    )
    
    # Re-order columns nicely for your presentation output
    return final_output[["Scientific_Name", "Common_Name", "Similarity_Score_Decimal", "Match_Percentage"]]

# ==========================================================
# TEST RUNNING THE UPDATED ENGINE
# ==========================================================
print("\n--- TEST RUN: Fetching Recommendations with Dual Scores ---")
sample_plant = df_encoded["Scientific_Name"].iloc[0] 
print(f"Target Plant Selected: {sample_plant}\n")

results = get_content_recommendations_dual_score(sample_plant, top_n=5)
print(results)

Cosine Similarity Lookup Repository successfully generated!

--- TEST RUN: Fetching Recommendations with Dual Scores ---
Target Plant Selected: Abelia × grandiflora

                    Scientific_Name  \
0                Wrightia religiosa   
1                    Wrightia dubia   
2     Acalypha wilkesiana 'Mosaica'   
3      Acalypha wilkesiana 'Moorea'   
4  Acalypha wilkesiana 'Java White'   

                                       Common_Name  Similarity_Score_Decimal  \
0  Wild Water Plum, Water Jasmine, Sacred Buddhist                       1.0   
1                                     Red Wrightia                       1.0   
2   Fire Fiji Plant, Beef Steak Plant, Fire Dragon                       1.0   
3         WiIkes' AcaIypha, CopperIeaf, Firedragon                       1.0   
4                         'Java White' Copper Leaf                       1.0   

  Match_Percentage  
0           100.0%  
1           100.0%  
2           100.0%  
3           100.0%  
4           1